# Signal shapes: a small CNN trained on CPU

An open-ended problem with a PyTorch model, in the SP5 submission format, with a CPU training budget (lesson u13-l03).

Each example is a signal of 64 numbers: noise plus one shape at a random position. There are 4 classes:

| label | shape |
|---|---|
| 0 | a bump up (a short positive peak) |
| 1 | a dip (a short negative peak) |
| 2 | a step up (low before some position, high after it) |
| 3 | a step down |

The position, width and height of the shape, the noise and a constant offset all vary from signal to signal. Predict the label.

Implement, in one file,

```python
def train(train_path, val_path, out_dir):
def predict(test_path):
```

with this contract (the one lesson u13-l04 describes under Practice):

- `train_path`, `val_path`: `.npz` files (u00-l04). `np.load(train_path)["X"]` is a float32 array of shape `(2000, 64)` and `["y"]` an int64 array of shape `(2000,)` with labels 0 to 3; the validation file has 500 signals. Save every artifact `predict` needs into the folder `out_dir`. This is the training notebook.
- `predict(test_path)`: a `.npz` file with only `"X"`, shape `(m, 64)`. Return a Python `list` of `m` Python `int`s, the predicted labels in order. This is `submission.py`.

The grader zips your file with the contents of `out_dir`, deletes the training data, unzips the package in a fresh folder, imports it from there, and calls `predict` with the current folder set somewhere else. Load artifacts inside `predict` (your file is also imported once before `train` runs, when no artifacts exist yet), with paths built from `Path(__file__).resolve().parent`.

**Saving a PyTorch model** (cheat sheet, u10-l08): define the model class in your file (in the real exam it can live in `submission.py` or in a separate `.py` file that goes in the ZIP). In `train`, save a checkpoint dict with `"model_config"` (the constructor arguments) and `"model_state_dict"` using `torch.save`. In `predict`, `torch.load(path, map_location="cpu")`, rebuild with `Net(**ckpt["model_config"])`, `load_state_dict`, `eval()`, and predict under `torch.no_grad()`.

**Scoring.** You pass if the accuracy on a hidden test of 1000 signals is at least **0.93**. The visible test prints the accuracy on a public test of 1000 signals and needs at least 0.92. For reference: logistic regression or a small fully connected network on the raw 64 numbers reaches about 0.88 to 0.90, because they see positions, not shapes. Other hidden tests check that two calls on the same file give the same predictions, and files with 1 and with 20,000 signals.

**Example.** If the test file holds 3 signals, `predict` returns something like `[2, 0, 3]`.

**Restrictions.** You may import `json`, `math`, `os`, `pathlib`, `time`, `copy`, `numpy` and `torch`. Read only the files you are given and the artifacts you saved. Set seeds so that your training is reproducible.

**Constraints.** `train` at most 60 s on CPU (this site adjusts the limit for the computer that grades your code, and runs PyTorch on one thread). Each `predict` call at most 30 s. Package at most 20 MB.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile submission.py
import copy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

HERE = Path(__file__).resolve().parent      # the folder this file is unzipped into on the grader


class SignalNet(nn.Module):
    # Your code: a small model; keep its constructor arguments in a config dict
    pass


def train(train_path, val_path, out_dir):
    d = np.load(train_path)
    X, y = torch.from_numpy(d["X"]), torch.from_numpy(d["y"])      # (2000, 64) float32, (2000,) int64
    # Your code: train (seeded), keep the best epoch on the validation file,
    # then torch.save a checkpoint (model_config + model_state_dict) into out_dir
    pass


def predict(test_path):
    X = torch.from_numpy(np.load(test_path)["X"])
    # Your code: rebuild the model from your checkpoint (path built from HERE), eval(), no_grad(),
    # return a list of Python ints 0 to 3
    return predictions


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import submission as submission
importlib.reload(submission)
# e.g. submission.predict(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9XAtz2za2/itYd2ZDemVGttPOVql7r5O6Xd/Ja2Lvbu5aHpYSIZlriuSSVGzH6/9+v+8A4EOSnTjt3nbGIkHg4JyD8waQ260k3hqpreXu/k4xfLY1UFsLXUdoul3/Uid1qtl2ksyzKFXVRVToaqQiVS2iNFUv37xRdRklmY5VnqmX7/7KUamuqjxzoNLhvkySZMtaV2jd+xavcTKbJdNlWt+g5RkadFaXfN4qSh0n01rG5PEy1WEWLQSJajlZJFWVADY+Yv78Ssdhsijysibgs61/VubbIqov+JtX/FvgLU0mhiCAwu80L274my0X5qHOy+nF1jmeJlEGcgjOvoVpnheEP4vSSrOvruqQkPJlHfLD7t6QzdHcYAG0dZpk8yAvdBbqLAY4N0Ug7MLHtmV6oaeXRZ5kjmYzmBOEk2U819LeEh9EZZ3MoiloJoZVviynwp+8TOYJlsnQXNZ7aKzLpb7D+yxJhfm3WxdRmWGBgkK4PZb//6LTQpeVqi+iWhVlPkn1Qs3LKGajYbDyfp2V+ULZ4a51+1c/GGfj7NB2V0mlpnlWYzmfLrOkrp7+wJ8fn1qoeC+S+MenpjeQCNRxrWI9gwxhfnBWzZbZtAadleLCx9IYbhvc6uhS40mPs5YfyoiJ+hVNvwbq6MPh63evjk5UmlQ1SdIGLIFVkOD8KqOsvl9myvuYVAmw8p+rE4KrVbnEtPqjLm9kEEj7OUrSZYmRWLdKK76qq6S+gA4AXhXNtboqkxoEq1leqhc6G6kqulFXRPcqqvA1z+YDlRGqiiFv6iKJY3RPsmJZV+SeWYXMspSy2bxQCshf+yryqgA1K9j6jXqZp9GEilepqNSqok6U5FVGVJ5U6vWeelfmA1XlKs2n0GFAhHhVarEEUyaaTAWLcqpjgs9pQj5MQP8FlpJilpcBZ9oLhlzcSM2XXH/PzFwBVqp36otSR7GqCo0FQ68yX84v0htISzpTwAVYTPOSjC41mV6rZEYE8Sraa+mwSkWmnLw7Ovop/Pnw5enb9+qAs5PgcTZNI8zOZfCOrqe6oKT4o3Gm8F+BT6YXBAoc17EHWYwHalHNXR/Mm+VkchbbFv7XLq7Hvi2QMCu8azcWFqozxq0QlbjTOgP9SVbVUTbV3vXAfA9OdVblpd8ZLtPqellm6jqIYYKnF54fTIsl/soye77prIVMdSzTHZVlXnagGJI7sLIiiKqoLKMb4N3SIZbbm+f1AEKZ8S/k82C8BcmGER5vOcwq8LpeFqn2SDn6+4EMtbgIT9HHdSIsf6Bm461bArzDgldmLnVbwe7o60JPa8jEbaf/HaZrMZumefUQZgNV1nl6sKt3vhuoyD7+2eELzboCyg2yUVXfgNBZmkdEjO0yZ/9Dh5i5oY8kXZnHe8gxBNju/iba3Cchr5kAC5JUMG9JjdHAI009X8FSUAq7367st+70tKRwF7SEGcck2awBbgWZ652mlolghmUX/1h28U9X8HRZgmPCCCK3iK75E00qb6521JXv+4QNQpNPWmm4PDWk8m1QlT6fxNCpj1EK+6C8NCrnNLuaAqtu8TMK9ud3fm/t9b+WUfoFUtlXKtOVSGexSPrAF44CiaiuS/t9vCXL0QLZIC4dAWn7/MfFoisagn9o+QDk1pSpy9UGAldltIoxhIHYCvEtDBjjW3z6Q9lDjb3Q1FuMpApFRWQ5+FTdux4yYWdF2iFr+LMVrLCA/SCUcC4Me+ggxpGVs7M+cZ2eOKh9TMMqiwp48vr/1zSneaYfaZQxpZPKa4qk+PYtHy4xFu3t4+Mx8B1I4OJ38ekT18OOAL1Y6+LgFJFeR8Asgqdg3yp66wD8ngthU0CY8u26x3gsixcN1OTr+R59Ed9NF6MXMt/Xsj0i2wVO1TcFdqLITFJ5E3+TrcHw1s5sHN7YiYl9lLXtazYpGBhrB8nOumvl4AiETvCSh4sllALRjTfLBmob5hSasb19ecUnh4kJHV8yF5tlMvOMASppQIyDKHAO9YoyhIzlfLlAUA4Xooo0murARZ3i9alPsItnrWZFvsSzEQdwxnPTE9EZnUfmrSJkPnNIAkEGwZXPoRqhjC4j+LdPSeGZMTJHT8DFnjTCZY3IDRIbyGKsGeAns4SRJUPmlpZvbhP1J7V791xd5eUlo/pIhFdx+RCPNvbSMhnItxxmEJ9kXqUZCgKrL+Xy8y6LYV27IfeGSFrSVPjDCrE3KCCDJNZ90oxE8G5yKE71FiBKpmBIkBYVSZI4/HU0VVNMwCHMviXqZrYigT2DA2QXOyZbZLPE4AjLEyZQywLqpOqrZKoJEBxK4KHzmdVg4Ow4WokIsWWGeInumxAwYolvgTrlh6REM4hJn3AK+gUzGTms455Y1UNGk4AQILuchQJGl87afEaU4nrzYMQotQ1HLL0Hyi6ieqq6aUMrkiFFsaQ2eHtducMiYpofDiykFVM0QU5z2TZFewN1uYfZ+vZRUB6sNPYo+TwrBE8yYk/YsNdpFjYgY/Niuth72NGNN7sEiR7VeS5yg+Q/rkfB3uwOUT6kTD+3WVcjxCJMUDsBpJYZ8/lbyxo7TnJIJHZuiTf8J1Mis0QaPINozRnAmiTz1q6TAKJ2PKigZTaHdurYLVib4GAl43wBbs8iiF/YdBxL8WdZRVGSh1Jh6FU5fmELNACYtcWDRIzGRZLG1LopKyNImBPqAnv+z8nbN8A1E82p6hgYiq4WN/UFWnaOlZtH/WBLHGGclD+qHzoFm+IG79DKf0tb/eNayg9l66T/WNjrOk0mTZP5QUuwrJO0bc6bRxa+mhdb72reKyneta831Up5wb2U8AuTaHrZtECKWauqTK2hmiYgM9ONuUPDZVLvpBrd1G7wXbBrY7pqxDrRgoXAgZgUZBfW+TjjV0QJuDwfZ26OYJakkGj36o23gHZeamaAttKCSDTY/mt2mbGGU+UpKyq5VAAwYVJw1YJtGyn+5ej9ETTH8iJ4h18vDFkGC0M/gOHicER1RVQCs3EGrgTsHEBadVl7QygDwgeC8QFwU+1LakHqG/jqf0UjdfRsuCeVsFd/P/zfk/DF4Zs3Rz8Bg9vxVmiLlGFIYkCYW0/zqq/11D4hzjdP03xRANfx1t04e/X27bvwzdufjk4AzoOsBD/n0FU+/B2Cq83jYXWTTZsPrxBDvgQM83aiOy8/Ichs337RGf1zXh5dF363xHJqajGfq7KEETK9hUcZWy5gb+C2dKOxkio6QN1qBCObaSiVz8qryimXuI76/va90XdGw3BW5GN+aZizqOZgzHOYRUaApsYnalPVT5rVkWDIlhLpqOoyMWXFnosyTujsfFM0W5cagZ1wCVKC/BqI9uPQkxvk59cSh7Iop9fjwzOL9gjSItVji/2IZrKS4TZFzjNjZ251wN8svxvxGb2R/NxZBG3Rmy4B/ApgmKEnK5Vw8ujs3CI6iZp28ZO1czlSDg2XlQDrtE+gD6Gpldp2ijhCRuhuEkNVGImVJlAwgAHB9PeY0TDoTiYs9YOcS4nddBpQsO0EUk3v8HwR1Ytl2m2ZSXUEppq2Gcy/itJLj4ux4ro7YTq7G3k2qcFqVtGEsuwYCLqjdQdmSyqIl71IOjEoX7C0YF4DzAZGMnL0Fog8YY6YzBH4QpA1C+GPNvvGzloEURzbOfz1zu3adPoFVZEmXG7Oebaze75hYLt4wbKIGXVvGNtNEtOH2fgzrN4qNVhQrJXw0RbeQX6rUBt4KSMsI/H8G7i4ykFAW+FCh3U9DmySgJWh9zHvq4YKnY9l+qHTnVXaVwRC8KCifRHUNx2L/BDAJL6fqs2dNs3YOqvVOUXxg6godBZ7ruJjBo23jMkz5v2/+vzaNIt4wRdJ9tZ6scPl/LCiD/JNbWVlQJDbfq+j+jVCxjWZNjbokcgBsY5AdkAySLSg7jX/LgowWmJEnq2iEOwX/DNneioG1uvM4/ubrcuf6FHAGAfq9slAPTFAnEZRAZ9gPuS98ycsxd5triduwp8liw3oVw73Ftg3zQ7iJMoq8nZ6YT2EhIKtg5Dys46QxH6QX/+5yUbTq+im2jEbschkEBcmDSAKpYWWZ+nNwM2JzKV1NZ6RAru/akIpBh03KtVzpOZXDJuU9AyCwO84MwRlKs65KAbwRSL5pVkFr+M0/yjOseOFDbrW+UIM/628ji/9o+rFhR0RwgyPFh1gUCnLH8L3Kn+D7ACy/1tWOMvdHM2WbLvKwLxPfGiUKMwLE0/2q23sbpRsRfW+hGBmlJCL/1YONlPWjt4oz0ROdoaz4XlXuDeQ/xgWtLM+RLwYtzWSpfXrCDZhkgh7j1A0U40p1KW+0JkksU3sKKN+X/otZXcPZOqzMgGY9CaUSNYDCCS6IbOpfkT/NtM7gmesq2mZSFah5OQDDJKyO9HSIXHb10zYoK6a/SrZrO9XmCYsq7i8NUDejOc6rCeeDsKw+YCUzwabBI5IcyZOfcZ5AAIrNWP6qV3Mx8yvIcFGpVcsmbBgRSsrtUkrdPjLcKwTtfeauBQiMOwt22kt+paZ3DGRHRLd2R9h9H93K7Pesb9j9gKRuNfkYrC0q1kuM9monH88a0JER8q9PffOOymx5RQD8APVdNl34TrkHu0sOARpHsWV5wkSTyk21AkpRWwRHnwWKxmeM3ZInwjRItPt4Lgh1cQDta4XJmEc21M5YzmCU7WvpjBjRNZm0pRENrzJsQRd+GcttHOic08q2ig6I1iqYHVmsDoXb8E30ZLzbjjrUjoXz1ZSd1ybtmslpGrhCTt5OKTyTPeulzdS4kqeMyCd5MFJzfLJ8VvHPFPkCcyPZ99Ojn85fPX+9cBm6bYrzFkoxV23C9yxZf2zTeTl3tDh0s+O5RBMW6wKeGyr1FA+sxgeMF0NsxyOuk44RenQPD49fn30Pnx/dPhq0GC3EuZUhab09EthAVtD1mZMZYfVSdky6VDUOURmRKMtzGGZOqZqZb7lZH06C0sm5NQe/6wOhMaYftUZP0vCcG4Ef50k0SFdBqwBhWYYLY//FYwb9msTtuzSK0lYQbTKcW6smaGRFlismljcOs8vZU9B3br1GAVDVoCd6bKzvIgq3VSITA2kLYm9eHU0HO62CDyKjPswthtDgiZ3YnhCLLcLZYOgB/xRo9ZroDcxytkVzgt5pkTJTryHBPzZcDgcnX+1Ist5x34dR5q65ztMH7dX7HY0QcvAfPI/v7gtr+Jcm6DJHLaDdyGIu27e/kgS5o/TyfGWOfRHLWxcRnMOsKHbniP8vO7NO8o3v0+XDDTbyx73s4Utl+WZLrQM7uigC+Kt2aOvYdDgIVGZGcPOJ9r2j1FZuTmCpNaLypPTM1nfK4hVdfv+3ECJgIY387sFL1gJbkJKSMIZV4JncceU/nIJu2WEgtGKnCLIHGkrBpfmOcmWurt+U+NgjT0cCZCzb0fnsN+yLwxsbZVaiXkcb1nQrnNnNn5lNZhpz6Y0xdpbs+nDpmEwvHvMntj6wYev8Dq/1fPYzbl1q/xVJs0uwlnLOWoqA+/1sxtrFrwdbGIPo+NirNM8m7t9mVWbrbwkM6fMJJofMIGYRaVyO4N+1wjYyWVzY6Wm/RsJbnBmeL3hrMqjfMnvhUv3nIHsV8Rf6ENaWI2AE14JKY69+zZqB2p/k3uz4ey5S8jKxrA92hU1A0ysesDU0Wtj14eiUjEqkzxPvTXEzDeCqnuSS3B1D1wzwiUo97oUOdY8Uy7hMdYtDJnfcKNs5LYHmO6Ynd3GWbR7uvaywkjJplUyu+EJ77JWuz9Z6YDwT5bc7IqTglt6ulBL9xBzHxFpZ6TMjjJzzES2utwRb3PNwRSWiO87JJPRnLGSZ245KHPF4dlAvWPmmUy1PzJXIrDiPBjOqpc7r89AhXE1TJapdFlf9ylBqt/bLzbzo8sCiOtUE4wce4+jOhqoZSZDwNocSM7A0wssRSpuzNXlZMeylqTVeZ1K2csVwE0+RywGAgEzNlDvbk5ZMjPH4InCOOMmuDvdndtD+PYEuRTU5KDNjTl7U11EgI8lm14gwlg/2X7PPnbe7keX7W50ddHrBN9aMJ7YvHUNdpiPQpZNct1OIHPd+87Q90+s3bfLO5ATEQMeSRisHDEZZ82FA4QI1tXr62hR0GVCKU/fHx6/CV8dvz4+RY/v5Cztu/dHPx2/PG1a96X19eGH8B/H78LXL3jgHQ2nh+9/OeL3YfD9vnsN/3Z8cgxraJr3xtkrgn02zsKXhy//wubbu8727CK61AxceicpaKfaQxNsYihaBhAoDS9ScfcbIs0TwAeZC4O468NjbXKM5pV/xsx6oJzdgeoIiCV0Oi8X3p8H6hVM3p8tFKKw6yKqqySWYkTbezf4FqYxGG7qDVb2+g4DAJUB632p6gZPfV14O8PgW7WtPK8GJkDQR8gpc/tqe1u5UzZiCmSMFFvQ+ccD9obr4CQE0ubYpmMFlYQWnd2cjQZSYTin9QL2/Ybd1Ya9cwSWxhztmB9OjsYd/gIrEmvm+kA/ifn+BMozkI2kfkiMnnXIfuX7a9/3N7HF1pk+uFPwIEEy//09RHg3nVYIwHfPrG0WwR2ovx1SwIwc7cHREOzAvn/LV/Lx9Ojk9MQElsVykiZTiWEZ8pmOuzJwX+JJe/1lU4dnfld0CzG2Jh9uD4Jjhss5S2qZMhK/vulu2s9sz/PO4agm9nbBvyRT0FpabIarXCpf0Lw/czLmXf5KQDDg8Wz7ZM2736QzcjLxQGyQ52xYsLiM+ezBHM+S6wM7P4xqCMsbtjtMtPYYLDCYLfHdxWp8JqCkiZg3YYTRAuSpIzLIik8ktGlFT9Nm2QTpjj7qTz36PhyIMJwNIao39rmpLjYjWi58OIDU2N58avpa9nRparxj50gG+/Rpu8hZmYWrYOAzvYobmsU1ya0V2ezB1xAGPrTtzVnDKpheEByn3VjOMsOrleG7jzrCB2Ey9/s8RrgtA31zlsjxx746UVk97PfwmcdOeL56f6nX4YsD6f59DkHaBcL3FKUb2Z7xtmHa5WLDZ66X/znu2l9/5TRn12NuOtF5zzFHwVxyITnnuNs555jU5robkyCESujXHGsc9o41eredyd3H3J5aa5X6k1Wtz2j1smBFQvRZRN0atADRihN2ifNs9BL8Iyl+xq/3yWjReOtKagaV+tRh8adAwkparsAdaBsgxpxyiQ665c1uVcWVGQoSb7cznaKV8zSfIO/n0bnV5BkGl3eUZB5vU2bt0ClaHApk5SmSio86rPNGzOFlQsbX1+1+APxUuGCZVQhmyQSRKX5CuXr0VO3q77rXzmx3LHsbKQ3c5qDlrdx3sT2NDLx+8Vxdarj4Ngy3J1tbKOzVsMoEnkG54JGjrsGY5NefX3MT0nds+P0rvGFp7QYWEy3ORqFZZiRNx73LP24zynWi22BTU511ffpW9LEFu1aWQkk5CrPBvY6avG6WPHNe6BGF9MaGNERsNNj3VfzaM0G/zRK2ZXGKlyN/JU0zBejR72QpV+IWnvYENcLvgVrlxn2xThs+2VxPQpwPAwULppuLgBvgMsrrhlxt9dNFEmbdZy6EC2+ZREjE1GYOu8Pt7e/9u80hhYPFCOHD7+zZH5aW38O9m7OiPNnVsJZu3hH1lY4cAvQmr39m2WjjQdKZ2ZQ2dkcqZ81Wte/L5lKvUDxzNVpvJeDrRHprd676VzpFUCDAlkpwMjGbLPz3DMyFehuumijSu5VJ7/zn9kqMqVjEaqLhcnQD54E7A2LBqpGiPsu5HrMXRbGJW8NtzpKb4zmUfxO0+/feTV0lBMuQmr0U6KO958IA3+D/h/IuUCeYUajIqoSXBWy8ynmFenyypaF7qBnLPzdhTi2ZIsvnjp5vfaW5YgmcibEOoK1eCXzG17uT8fjsbLjz/fPz7YVZdf41BVf/bLQ/HJ63uH6j4jIvAKhc0D6BP2luSqHVF/P0oVCRGP6nosWebQqWWZpkl95qJNmvs3x5LLmuAhvDSnNcZWGEfnNE2b8LLkyCTPkcwIPtm2dzl6ksVzlg473ZSL0z11AIqj8TDISZ6OBA8fmD/7mp3Ig7903+CQ7GjLcGwJ2rqLYram77fexf9COQ1bt9QsdHIRueQtRpSJ5/5J/9jahRA24Tc4H5I3TzudLR9ML9sxUt7YQ3pHXaX71N5C5hCkbdGwvT6RJR1o3xjFnneCy8JAthUs04k337/jkdOS3hrXlWAUFO3/hyjtHzO9N1C4LdUgawoMftI7NSPukV0We9jyPlRqpbPI2CZ7OVf3iAE/wIYnpVQ2H1Z8CQ5ViK/Erd9gf3b2ELYTYS4PMXENcr/dyL66ANMb4AR/FH/McGaofuBjR5tdRYYLkG2UX1Q7PgK/idnw3PjcW0NEmAtLL04632QmRD0mRzT7NPJF3NxU5ekVsYH9Rnh2euAzf/IgRc+VVuS/irI9WcLov/npKmO+lq78jM1TsXa33yA75YfJgFwo0O+giGPrwmiNhdcGem7WIAgvc3sBxJWLXKaFdGDKXc971/5pi7wq6z0a4c7IrsAUC6XLm4Z66yOWZtYHJ3xN5wgHl6Zmvr7u7/AJ99MYE=")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
